# 🏥 التنبؤ بتكلفة التأمين الصحي
**Insurance Cost Prediction**

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [1]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "ml/b9_insurance_cost"          # مسار المشروع داخل portfolio/
PKGS    = []          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

✓ جاهز — D:\dataanalyst\portfolio\ml\b9_insurance_cost


## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| الموضوع | المصدر المقترح |
|---|---|
| Linear Regression + Regularization | ISLR Ch.3 & Ch.6 / Géron Ch.4 |
| Feature Interactions | Géron Ch.2 |
| Gradient Boosting (XGBoost) | Géron Ch.7 |
| Regression Evaluation (RMSE, MAE, R²) | ISLR Ch.2 |

## 🎯 بيُستخدم في إيه (استخدام واقعي)
- **شركات التأمين** بتستخدم النماذج دي لتسعير البوالص بدقة.
- **Actuaries** بيحتاجوا يفهموا إيه العوامل اللي بترفع التكلفة (التدخين، السمنة، العمر).
- **فرق الـ underwriting** بتستخدم الموديل عشان تحدد المخاطر.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

## 1️⃣ تحميل البيانات والاستكشاف (EDA)

In [3]:
df = pd.read_csv("data/insurance.csv")
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:", df.isnull().sum().sum())
print("\nStatistics:")
df.describe().round(1)

Shape: (1500, 7)

Data types:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

Missing values: 0

Statistics:


,age,bmi,children,charges
count,1500.0,1500.0,1500.0,1500.0
mean,41.3,28.4,1.5,22164.9
std,13.5,6.0,1.4,10964.7
min,18.0,15.0,0.0,4468.7
25%,29.0,24.3,0.0,14807.6
50%,42.0,28.6,1.0,18707.4
75%,53.0,32.5,2.0,23963.5
max,64.0,47.6,5.0,55951.9


In [4]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Target distribution
axes[0, 0].hist(df["charges"], bins=30, color="#3498db", edgecolor="white")
axes[0, 0].set_title("توزيع تكلفة التأمين (charges)")
axes[0, 0].set_xlabel("Charges ($)")

# Age vs charges
axes[0, 1].scatter(df["age"], df["charges"], alpha=0.4, s=15, c=df["smoker"].map({"yes":"red","no":"#3498db"}))
axes[0, 1].set_title("العمر vs التكلفة (أحمر=مدخن)")
axes[0, 1].set_xlabel("Age")
axes[0, 1].set_ylabel("Charges")

# BMI vs charges
axes[0, 2].scatter(df["bmi"], df["charges"], alpha=0.4, s=15, c=df["smoker"].map({"yes":"red","no":"#3498db"}))
axes[0, 2].set_title("BMI vs التكلفة (أحمر=مدخن)")
axes[0, 2].set_xlabel("BMI")

# Smoker boxplot
df.boxplot(column="charges", by="smoker", ax=axes[1, 0])
axes[1, 0].set_title("التكلفة حسب التدخين")
plt.sca(axes[1, 0]); plt.xlabel("Smoker"); plt.ylabel("Charges")

# Children boxplot
df.boxplot(column="charges", by="children", ax=axes[1, 1])
axes[1, 1].set_title("التكلفة حسب عدد الأولاد")

# Region boxplot
df.boxplot(column="charges", by="region", ax=axes[1, 2])
axes[1, 2].set_title("التكلفة حسب المنطقة")

plt.suptitle("")
plt.tight_layout()
plt.savefig("data/eda_overview.png", dpi=120, bbox_inches="tight")
plt.show()

## 2️⃣ هندسة الميزات (Feature Engineering)

In [5]:
df["bmi_smoker"] = df["bmi"] * (df["smoker"] == "yes").astype(int)
df["age_sq"] = df["age"] ** 2
df["overweight"] = (df["bmi"] >= 30).astype(int)

print("New features added: bmi_smoker, age_sq, overweight")
print("Shape:", df.shape)
df.head()

New features added: bmi_smoker, age_sq, overweight
Shape: (1500, 10)


,age,sex,bmi,children,smoker,region,charges,bmi_smoker,age_sq,overweight
0,56,female,27.8,1,no,southwest,18916.38,0.0,3136,0
1,46,male,29.6,1,no,northwest,18995.93,0.0,2116,0
2,32,female,32.8,1,no,northwest,15384.02,0.0,1024,1
3,60,male,35.4,3,no,southwest,26644.97,0.0,3600,1
4,25,female,27.3,0,yes,northwest,36037.52,27.3,625,0


## 3️⃣ تجهيز البيانات والتقسيم

In [6]:
target = "charges"
num_cols = ["age", "bmi", "children", "bmi_smoker", "age_sq", "overweight"]
cat_cols = ["sex", "smoker", "region"]

X = df[num_cols + cat_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_cols)
])

Train: 1200, Test: 300


## 4️⃣ تدريب ومقارنة النماذج

In [7]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge (alpha=1)": Ridge(alpha=1),
    "Random Forest (200)": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                 random_state=42, verbosity=0),
}

results = []
best_score, best_name, best_pipe = -1, None, None

for name, model in models.items():
    pipe = Pipeline([("pre", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R²": r2})
    if r2 > best_score:
        best_score, best_name, best_pipe = r2, name, pipe
    print(f"{name:25s}  RMSE={rmse:,.0f}  MAE={mae:,.0f}  R²={r2:.4f}")

results_df = pd.DataFrame(results).sort_values("R²", ascending=False)
print(f"\n🏆 Best: {best_name} (R²={best_score:.4f})")

Linear Regression          RMSE=2,522  MAE=2,057  R²=0.9494
Ridge (alpha=1)            RMSE=2,538  MAE=2,071  R²=0.9487


Random Forest (200)        RMSE=2,741  MAE=2,180  R²=0.9402


Gradient Boosting          RMSE=2,739  MAE=2,183  R²=0.9403


XGBoost                    RMSE=2,681  MAE=2,139  R²=0.9428

🏆 Best: Linear Regression (R²=0.9494)


## 5️⃣ تقييم النموذج الأفضل

In [8]:
y_pred_best = best_pipe.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, alpha=0.4, s=15, color="#2ecc71")
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], "r--", linewidth=2, label="Perfect fit")
axes[0].set_xlabel("Actual Charges ($)")
axes[0].set_ylabel("Predicted Charges ($)")
axes[0].set_title(f"Actual vs Predicted — {best_name}")
axes[0].legend()

# Residuals
residuals = y_test - y_pred_best
axes[1].hist(residuals, bins=30, color="#e74c3c", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="black", linestyle="--")
axes[1].set_title("توزيع الأخطاء (Residuals)")
axes[1].set_xlabel("Error ($)")

plt.tight_layout()
plt.savefig("data/evaluation.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Residual mean: ${residuals.mean():,.0f}, std: ${residuals.std():,.0f}")

Residual mean: $-116, std: $2,524


## 6️⃣ أهمية الميزات (Feature Importance)

In [9]:
feature_names = num_cols + list(
    best_pipe.named_steps["pre"].transformers_[1][1].get_feature_names_out(cat_cols))

inner = best_pipe.named_steps["model"]
if hasattr(inner, "feature_importances_"):
    importances = inner.feature_importances_
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    fi.tail(10).plot.barh(ax=ax, color="#3498db")
    ax.set_title(f"Feature Importance — {best_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig("data/feature_importance.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    coefs = inner.coef_
    fi = pd.Series(np.abs(coefs), index=feature_names).sort_values(ascending=True)
    print("Top features (|coefficient|):")
    print(fi.tail(10).to_string())


Top features (|coefficient|):
region_northwest       79.357070
region_southwest       95.515754
region_southeast      108.893788
age_sq                111.448027
children              715.280191
sex_male              870.982658
bmi                  1117.940038
bmi_smoker           2269.031504
age                  3110.585891
smoker_yes          18819.190943


## 7️⃣ مقارنة النماذج

In [10]:
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(results_df))
bars = ax.bar(x, results_df["R²"], color=["#e74c3c" if n == best_name else "#3498db"
              for n in results_df["Model"]], edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(results_df["Model"], rotation=25, ha="right")
ax.set_ylabel("R² Score")
ax.set_title("مقارنة أداء النماذج")
ax.set_ylim(results_df["R²"].min() - 0.05, 1.0)
for i, v in enumerate(results_df["R²"]):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontweight="bold", fontsize=9)
plt.tight_layout()
plt.savefig("data/model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 8️⃣ الخلاصة والتوصيات

In [11]:
print("=" * 50)
print("📊 ملخص النتائج")
print("=" * 50)
print(f"\n🏆 أفضل نموذج: {best_name}")
print(f"   R² = {best_score:.4f}")
print(f"   RMSE = ${root_mean_squared_error(y_test, y_pred_best):,.0f}")
print(f"   MAE = ${mean_absolute_error(y_test, y_pred_best):,.0f}")
print(f"\n📌 أهم العوامل المؤثرة على التكلفة:")
print("   1. التدخين (أكبر عامل بفارق كبير)")
print("   2. العمر (علاقة طردية)")
print("   3. BMI (خصوصاً للمدخنين — interaction)")
print("   4. عدد الأولاد")
print("\n✅ التحليل اكتمل بنجاح!")

📊 ملخص النتائج

🏆 أفضل نموذج: Linear Regression
   R² = 0.9494
   RMSE = $2,522
   MAE = $2,057

📌 أهم العوامل المؤثرة على التكلفة:
   1. التدخين (أكبر عامل بفارق كبير)
   2. العمر (علاقة طردية)
   3. BMI (خصوصاً للمدخنين — interaction)
   4. عدد الأولاد

✅ التحليل اكتمل بنجاح!
